# Three-Body Capture Walkthrough

This notebook walks through selected $r=8$ kpc entries in Table 2 of Saricaoglu and Daylan, *Captured Primordial Black Holes in Microlensing Surveys*. The values and uncertainties are transcribed from the manuscript because the 120,000-system simulation archives are not included with Captus.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np

import captus
from captus.paths import get_visuals_path
from tdpy.verbosity import print

print(f"Python {sys.version.split()[0]} | Captus {captus.__version__}")

In [ ]:
# Values and 2-sigma uncertainties transcribed from the manuscript's Table 2.
# The original encounter archives are not bundled with Captus.
paper_tables = {
    "sun-jupiter": {
        "title": "Sun-Jupiter",
        "mass_msun": np.array([1e-8, 1e-4, 1e-3, 1e-2]),  # [M_sun]
        "capture_rate": np.array([0.27, 0.31, 0.76, 7.46]),  # [Myr^-1]
        "capture_rate_err": np.array([0.01, 0.01, 0.02, 0.20]),  # [Myr^-1]
        "neq_per_density": np.array([0.15, 0.17, 4.17, 41.79]),  # [AU^3]
        "neq_per_density_err": np.array([0.01, 0.01, 0.14, 1.34]),  # [AU^3]
        "neq_full": np.array([2.1e-11, 2.4e-15, 5.9e-15, 5.9e-15]),
        "neq_full_err": np.array([0.1e-11, 0.1e-15, 0.2e-15, 0.2e-15]),
        "neq_bound": np.array([2.0e-13, 4.5e-18, 1.3e-18, 1.5e-19]),
        "neq_bound_err": np.array([0.1e-13, 0.2e-18, 0.0, 0.0]),
    },
    "beta-pictoris": {
        "title": "Beta Pictoris",
        "mass_msun": np.array([1e-8, 1e-4, 1e-3, 1e-2]),  # [M_sun]
        "capture_rate": np.array([13, 14, 30, 173]),  # [Myr^-1]
        "capture_rate_err": np.array([1, 1, 3, 13]),  # [Myr^-1]
        "neq_per_density": np.array([5, 4, 122, 816]),  # [AU^3]
        "neq_per_density_err": np.array([1, 1, 14, 80]),  # [AU^3]
        "neq_full": np.array([7.5e-10, 5.8e-15, 1.7e-14, 1.1e-14]),
        "neq_full_err": np.array([2.1e-10, 1.6e-15, 0.2e-14, 0.1e-14]),
        "neq_bound": np.array([7.3e-12, 1.3e-18, 4.3e-19, 3.8e-20]),
        "neq_bound_err": np.array([2.0e-12, 0.3e-18, 0.5e-19, 0.4e-20]),
    },
}


## Read the selected entries

The masses below are selected from the manuscript's logarithmic PBH mass grid. At 8 kpc, the capture rate and equilibrium capacity increase strongly as the PBH mass approaches the host planet's mass. The abundance-weighted counts also depend on the assumed total or observationally allowed PBH fraction.

In [ ]:
system_name = "sun-jupiter"
selected = paper_tables[system_name]
mass_grid = selected["mass_msun"]
capture_rate_grid = selected["capture_rate"]

print(f"Mass-grid shape={mass_grid.shape}, dtype={mass_grid.dtype}")
print(f"Capture-rate shape={capture_rate_grid.shape}, dtype={capture_rate_grid.dtype}")
print("Selected mass points:", mass_grid[[0, -2, -1]])
np.testing.assert_equal(mass_grid.shape, capture_rate_grid.shape)
np.testing.assert_equal(mass_grid.shape, selected["neq_per_density"].shape)
assert np.all(np.diff(mass_grid) > 0)


In [ ]:
def plot_table2(system_name):
    data = paper_tables[system_name]
    figure, axes = plt.subplots(
        3, 1, figsize=(7, 8), sharex=True, layout="constrained"
    )
    axes[0].errorbar(
        data["mass_msun"], data["capture_rate"],
        yerr=data["capture_rate_err"], fmt="o", color="tab:blue", capsize=3,
    )
    axes[0].set_ylabel(r"$R_{\\rm cap}$ [Myr$^{-1}$]")
    axes[1].errorbar(
        data["mass_msun"], data["neq_per_density"],
        yerr=data["neq_per_density_err"], fmt="s", color="tab:orange", capsize=3,
    )
    axes[1].set_ylabel(r"$N_{\\rm eq}/n_\\infty$ [AU$^3$]")
    axes[2].errorbar(
        data["mass_msun"], data["neq_full"], yerr=data["neq_full_err"],
        fmt="o", color="tab:green", capsize=3, label=r"$f_{\\rm total}$",
    )
    axes[2].errorbar(
        data["mass_msun"], data["neq_bound"], yerr=data["neq_bound_err"],
        fmt="s", color="tab:red", capsize=3, label=r"$f_{\\rm bound}$",
    )
    axes[2].set_ylabel(r"$N_{\\rm eq}$")
    axes[2].set_xlabel(r"PBH mass [$M_\\odot$]")
    axes[2].set_yscale("log")
    axes[2].legend(frameon=False)

    for axis in axes:
        axis.set_xscale("log")
        axis.grid(False)
    figure.suptitle(f"{data['title']} capture results at 8 kpc")

    output_path = get_visuals_path() / f"captus_{system_name}_table2.png"
    print(f"Writing to visuals/{output_path.name}...")
    figure.savefig(output_path, dpi=180)
    display(figure)
    plt.close(figure)


for system_name in paper_tables:
    plot_table2(system_name)

## Read the trends

The paper reports that the Sun-Jupiter capture rate rises from about 0.3 Myr$^{-1}$ for low PBH masses to several Myr$^{-1}$ near Jupiter's mass. The equilibrium capacity rises at the same time because the captured systems live longer. The abundance-weighted counts remain small because the allowed PBH fraction is restrictive.

The Beta Pictoris entries are larger because that host configuration has different encounter and survival properties. These plotted points are the manuscript's reported Monte Carlo estimates, not a rerun of its simulation ensemble.

Figures are saved directly under Captus's ignored `visuals/` directory. Export this notebook from Jupyter, or run `jupyter nbconvert --to html examples/paper_threebody_table.ipynb` from the repository root.